# CISC 886 — Section 5: Model Fine-Tuning

**Model:** `unsloth/Llama-3.2-3B-Instruct`  
**Technique:** QLoRA (4-bit quantisation + LoRA adapters)  
**Library:** Unsloth + HuggingFace TRL  
**Hardware:** Google Colab T4 (16 GB VRAM) or EC2 `g4dn.xlarge`  

This notebook:
1. Installs dependencies
2. Loads the 4-bit quantised Llama 3.2 3B Instruct model via Unsloth
3. Attaches LoRA adapters with PEFT
4. Fine-tunes on the preprocessed Alpaca dataset
5. Saves the merged model and exports it to GGUF for Ollama

In [1]:
# ============================================================
# 1. Install dependencies
# Unsloth's pip install also brings in the correct versions of
# transformers, trl, peft, bitsandbytes, and xformers.
# ============================================================
!pip install --quiet unsloth
!pip install --quiet datasets
# Confirm GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [2]:
# ============================================================
# 2. Load the base model in 4-bit (QLoRA)
#
# FastLanguageModel.from_pretrained does three things at once:
#   a) Downloads the model weights from Unsloth's HF repo
#   b) Applies 4-bit NF4 quantisation via bitsandbytes
#   c) Patches attention with Unsloth's optimised CUDA kernels
#
# max_seq_length: 2048 — covers most Alpaca samples; longer contexts
# cost quadratically more memory so we keep the default.
# dtype=None: Unsloth auto-selects bfloat16 on Ampere GPUs (A100),
# float16 on older (T4). Set explicitly if needed.
# load_in_4bit=True: enables QLoRA — critical for fitting on T4 VRAM.
# ============================================================
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype        = None,       # auto-detect
    load_in_4bit = True,       # QLoRA
)
print('Base model loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded.


In [3]:
# ============================================================
# 3. Attach LoRA adapters
#
# get_peft_model wraps the frozen base model with trainable
# rank-decomposition matrices (LoRA) on the attention and MLP layers.
#
# r=16: LoRA rank.  Higher rank = more capacity but more memory.
#   16 is a good default for instruction tuning on a 3B model.
# lora_alpha=16: scaling factor (usually set equal to r).
# lora_dropout=0: Unsloth recommends 0 for its optimised kernels.
# target_modules: the attention projection matrices and MLP gates
#   are the typical LoRA targets for Llama-family models.
# use_gradient_checkpointing='unsloth': trades compute for memory,
#   allowing larger effective batch sizes on 16 GB VRAM.
# random_state=42: reproducible weight initialisation.
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

# Show trainable vs frozen parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of total)')

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable parameters: 24,313,856 (1.30% of total)


In [4]:
# ============================================================
# 4. Load and format the dataset
#
# We load the tatsu-lab/alpaca dataset from Hugging Face.
# (In the full pipeline the preprocessed Parquet files from S3
# would be downloaded here; for the Colab notebook we use the
# HuggingFace version directly for convenience.)
#
# The 'formatting_prompts_func' converts each row into the
# Alpaca prompt template that the model was designed for.
# 'EOS_TOKEN' is appended so the model learns to stop generating
# at the end of the response rather than continuing indefinitely.
# ============================================================
from datasets import load_dataset

dataset = load_dataset('tatsu-lab/alpaca', split='train')

EOS_TOKEN = tokenizer.eos_token

ALPACA_PROMPT_WITH_INPUT = (
    'Below is an instruction that describes a task, paired with an input that '
    'provides further context. Write a response that appropriately completes the '
    'request.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Input:\n{input}\n\n'
    '### Response:\n{output}'
)
ALPACA_PROMPT_NO_INPUT = (
    'Below is an instruction that describes a task. Write a response that '
    'appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Response:\n{output}'
)

def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(
        examples['instruction'], examples['input'], examples['output']
    ):
        if inp and inp.strip():
            text = ALPACA_PROMPT_WITH_INPUT.format(
                instruction=instruction, input=inp, output=output
            )
        else:
            text = ALPACA_PROMPT_NO_INPUT.format(
                instruction=instruction, output=output
            )
        texts.append(text + EOS_TOKEN)
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print('Dataset formatted. Example prompt (first 300 chars):')
print(dataset[0]['text'][:300])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

Dataset formatted. Example prompt (first 300 chars):
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body act


In [5]:
# ============================================================
# 5. Configure and run training
#
# SFTTrainer (Supervised Fine-Tuning Trainer) from TRL wraps
# HuggingFace Trainer with sane defaults for instruction tuning.
#
# Key hyperparameters (see report table):
#   learning_rate    : 2e-4 — standard starting point for LoRA;
#                      higher rates cause instability with adapters.
#   per_device_batch : 2    — max that fits on T4 with grad checkpointing.
#   gradient_accum   : 4    — effective batch = 2×4 = 8 samples.
#   num_train_epochs : 1    — one full pass through 52K samples; enough
#                      for a clear improvement without overfitting on this
#                      general-purpose dataset.
#   warmup_steps     : 5    — short warmup; dataset is large so the model
#                      converges quickly.
#   lr_scheduler     : linear decay — reduces LR smoothly to zero.
#   fp16 / bf16      : auto-selected based on GPU capability.
#   logging_steps    : 10   — record loss every 10 steps for the loss curve.
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size   = 2,
        gradient_accumulation_steps   = 4,
        warmup_steps                  = 5,
        num_train_epochs              = 1,
        learning_rate                 = 2e-4,
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = 'adamw_8bit',
        weight_decay                  = 0.01,
        lr_scheduler_type             = 'linear',
        seed                          = 42,
        output_dir                    = 'outputs',
        report_to                     = 'none',
    ),
)

trainer_stats = trainer.train()
print('Training complete.')
print(f'Total training time: {trainer_stats.metrics["train_runtime"]:.1f}s')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/52002 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52,002 | Num Epochs = 1 | Total steps = 6,501
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.903382
20,1.252518
30,1.209973
40,1.257734
50,1.068202
60,1.053295
70,1.123594
80,1.092424
90,1.164960
100,1.224716


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-4000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-4500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-5000/tokenizer_config.json.
Unsloth: Restored add

Training complete.
Total training time: 10108.8s


In [6]:
# ============================================================
# 6. Before / After comparison
#
# We generate responses from the fine-tuned model with and
# without the LoRA weights loaded to demonstrate the effect
# of fine-tuning.  The 'FastLanguageModel.for_inference' call
# enables Unsloth's 2x-faster inference mode.
# ============================================================
FastLanguageModel.for_inference(model)

test_prompt = (
    'Below is an instruction that describes a task. Write a response that '
    'appropriately completes the request.\n\n'
    '### Instruction:\n'
    'Explain why the sky is blue in simple terms.\n\n'
    '### Response:\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens = 200,
    temperature    = 0.7,
    do_sample      = True,
)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- Fine-tuned model response ---')
print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

--- Fine-tuned model response ---
The sky appears blue because of the way that sunlight interacts with the atmosphere. When sunlight enters the Earth's atmosphere, it is scattered in all directions. Blue light is scattered more than other colors, so it reaches our eyes from all directions. This is why the sky appears blue.


In [ ]:
# ============================================================
# 7. Save the fine-tuned model and export to GGUF
#
# save_pretrained_gguf converts the merged (base + LoRA) weights
# to GGUF format, which Ollama requires.
#
# quantization_method='q4_k_m' applies 4-bit k-quant quantisation
# inside the GGUF, keeping the file small (~2 GB) while preserving
# most quality.  This is the recommended default in the Unsloth docs.
#
# After this cell, upload the GGUF file to your EC2 instance with:
#   scp -i ~/.ssh/id_rsa q1abc-llama3.2-3b-alpaca.gguf \
#       ubuntu@<EC2_PUBLIC_IP>:~/models/
# ============================================================
model.save_pretrained_gguf(
    '20596365-llama3.2-3b-alpaca',
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('GGUF export complete: 20596365-llama3.2-3b-alpaca-Q4_K_M.gguf')

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in 20596365-llama3.2-3b-alpaca/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:36<00:00, 138.19s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:43<00:00, 82.00s/it]


Unsloth: Merge process complete. Saved to `/content/20596365-llama3.2-3b-alpaca`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['20596365-llama3.2-3b-alpaca_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['20596365-llama3.2-3b-alpaca_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model 20596365-llama3.2-3b-alpaca_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to 20596365-llama3.2-3b-alpaca_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f 20596365-llama3.2-3b-alpaca_gguf/Modelfile
GGUF export complete: q1abc-llama3.2-3b-alpaca-Q4_K_M.gguf
